In [1]:
%run 0_1_load_paths.ipynb

In [2]:
import os
import subprocess
import glob

import momapy_kb.neo4j.core
import neo4j_dm.core

import credentials
import commute_dm.utils

We connect to the DB and we delete all existing nodes and relationships:

In [3]:
momapy_kb.neo4j.core.connect(credentials.NEO4J_URI, credentials.NEO4J_USERNAME, credentials.NEO4J_PASSWORD)

<div class="alert alert-danger">We delete all the data from the DB</div>

In [4]:
momapy_kb.neo4j.core.delete_all()

## Adding AD BEL KG to the Neo4j database

We import the cypher dump:

In [5]:
command = [
    "cat", AD_KG_CYPHER_FILE, "|",
    "cypher-shell",
    "-a", credentials.NEO4J_URI,
    "-u", credentials.NEO4J_USERNAME,
    "-p", credentials.NEO4J_PASSWORD,
    "-d", credentials.NEO4J_DATABASE
]

In [6]:
subprocess.run(" ".join(command), shell=True)

CompletedProcess(args='cat ../../data/ad_kg/cypher/ad_kg.cypher | cypher-shell -a localhost -u neo4j -p neofourj -d neo4j', returncode=0)

We make the Collection, Entry and Model nodes:

In [7]:
query = f"""
    MERGE
        (collection:Collection {{name: 'AD_KG_BEL'}})-[:HAS_ENTRY]->(collection_entry:CollectionEntry {{file_path: '{AD_KG_CYPHER_FILE}'}})-[:HAS_MODEL]->(model:BELModel)
    RETURN
        collection, collection_entry, model
"""
_ = momapy_kb.neo4j.core.run(query)

In [8]:
query = """
    MATCH (n)
    WHERE NOT n:Collection AND NOT n:CollectionEntry AND NOT n:BELModel
    SET n:BELModelElement
    RETURN n
"""
_ = momapy_kb.neo4j.core.run(query)

We link each node of the dump to the newly created model:

In [9]:
query = """
    MATCH (model_element:BELModelElement), (model:BELModel)
    MERGE (model)-[:HAS_NODE]->(model_element)
    RETURN model, model_element
"""
_ = momapy_kb.neo4j.core.run(query)

We extract subgraph information from relationships and make Subgraph nodes

In [ ]:
query = """
    MATCH (n)-[r]->(m), (model:BELModel)
    UNWIND r.annotationSubgraph AS subgraph
    MERGE (model)-[:HAS_SUBGRAPH]->(subgraph_node:Subgraph {name: subgraph})
    RETURN subgraph_node
"""
_ = momapy_kb.neo4j.core.run(query)

We add nodes to subgraphs:

In [ ]:
query = """
CALL () {
    MATCH (n)-[r]->(m)
    UNWIND r.annotationSubgraph AS subgraph
    RETURN n AS n, subgraph AS subgraph
    UNION
    MATCH (m)-[r]->(n)
    UNWIND r.annotationSubgraph AS subgraph
    RETURN n AS n, subgraph AS subgraph
}
MATCH (subgraph_node:Subgraph)
WHERE subgraph_node.name = subgraph
MERGE (subgraph_node)-[:HAS_NODE]->(n)
RETURN subgraph_node, n
"""
_ = momapy_kb.neo4j.core.run(query)

We make a special Subgraph node with name "main_model" for nodes which do not belong to a subgraph (in order to have uniform queries later):

In [ ]:
query = """
    MATCH (model:BELModel)
    MERGE (model)-[:HAS_SUBGRAPH]->(subgraph_node:Subgraph {name: 'main_model'})
    RETURN subgraph_node
"""
_ = momapy_kb.neo4j.core.run(query)

We add all nodes that do not belong to a subgraph to the "main_model" Subgraph node:

In [ ]:
query = """
    MATCH (n:BELModelElement), (subgraph_node:Subgraph {name: 'main_model'})
    WHERE NOT EXISTS {(n)<-[r:HAS_NODE]-(s:Subgraph)}
    MERGE (subgraph_node)-[:HAS_NODE]->(n)
    RETURN n
"""
_ = momapy_kb.neo4j.core.run(query)

## Adding the COVID-19 DM and PD DM to the Neo4j database

We save the collection to the DB:

In [5]:
#collection_names_and_input_dir_paths = [("COVID_DM_CD", COVID_DM_CD_PICKLE_BUILD_DIR), ("PD_DM_CD", PD_DM_CD_PICKLE_BUILD_DIR)]

In [6]:
collection_names_and_input_dir_paths = [("COVID_DM_CD", COVID_DM_CD_BUILD_DIR), ("PD_DM_CD", PD_DM_CD_BUILD_DIR)]

In [ ]:
collection_names_and_file_paths = [
    (
        collection_name_and_input_dir_path[0],
        glob.glob(os.path.join(collection_name_and_input_dir_path[1], "*.xml"))
    )
    for collection_name_and_input_dir_path in collection_names_and_input_dir_paths
]
neo4j_dm.core.save_collections_from_file_paths(collection_names_and_file_paths)

Received notification from DBMS server: <GqlStatusObject gql_status='01N51', status_description='warn: unknown relationship type. The relationship type `HAS_OUTSIDE` does not exist. Verify that the spelling is correct.', position=<SummaryInputPosition line=1, column=106, offset=105>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_status_parameters': {'reltype': 'HAS_OUTSIDE'}, '_severity': 'WARNING', '_position': {'offset': 105, 'line': 1, 'column': 106}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: 'MATCH (compartment) WHERE elementId(compartment)=$compartment_1 WITH compartment MATCH (compartment)-[r1:`HAS_OUTSIDE`]->(outside_r1:Compartment) WITH outside_r1 RETURN count(outside_r1)'
Received notification from DBMS server: <GqlStatusObject gql_status='01N51', statu